# AksaraLine — build the corpus and train the recognizers on a GPU

Thin driver for Kaggle/Colab. Everything substantive lives in `aksara_seq/`;
this notebook only wires up paths and calls the scripts.

**Attach the cleaned character corpus** so that `DATA_ROOT` below contains
`Sunda/`, `Jawa/`, `Bali/`, `Lontara/`.

Order: glyph pool → rendered corpus → verify → train matrix. The first two are
CPU-bound (~10 and ~50 min); only the last needs the GPU, so if you are
resuming a session with the corpus already written, skip to step 4.

In [ ]:
import os, sys, subprocess, zipfile, time
from pathlib import Path

BRANCH  = 'aksara-seq'
REPO    = Path('/kaggle/working/aksara_OCR')
DATASET = Path('/kaggle/input/aksara-clean-4')   # attached dataset
DATA_ROOT = Path('/kaggle/working/data/clean')   # where the scripts end up
BUILD   = Path('/kaggle/working/build')

if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/phoenixfin/aksantara-ocr.git',
                    str(REPO)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO / 'aksara_seq' / 'src'))

# The dataset ships one zip per script. Kaggle sometimes auto-extracts
# archives and sometimes does not, so handle both rather than assume.
SCRIPTS = ['Sunda', 'Jawa', 'Bali', 'Lontara']
DATA_ROOT.mkdir(parents=True, exist_ok=True)
for s in SCRIPTS:
    target = DATA_ROOT / s
    if target.is_dir():
        continue
    if (DATASET / s).is_dir():                       # already extracted
        os.symlink(DATASET / s, target)
        continue
    zf = DATASET / f'{s}.zip'
    assert zf.exists(), f'neither {DATASET/s} nor {zf} found'
    t0 = time.time()
    with zipfile.ZipFile(zf) as z:
        z.extractall(DATA_ROOT)
    print(f'extracted {s} in {time.time()-t0:.0f}s')

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
for s in SCRIPTS:
    n = sum(1 for _ in (DATA_ROOT / s).rglob('*') if _.is_file())
    print(f'  {s:9s} {n} files')


## 1. Glyph pool

Verifies the (onset × vowel) grids against the known class counts before
writing anything. `--verify-only` reports and exits if you just want the check.

In [ ]:
GLYPHS = BUILD / 'glyphs'
!python aksara_seq/scripts/01_build_glyph_pool.py \
    --data-root {DATA_ROOT} --out {GLYPHS} --workers 4 --no-hash

## 2. Look before you generate

Renders a few lines per style with bounding boxes drawn. Worth 30 seconds —
layout problems are obvious here and invisible in a loss curve.

In [ ]:
!python aksara_seq/scripts/02_render_corpus.py --glyph-cache {GLYPHS} \
    --preview 2 --boxes --preview-out {BUILD}/preview.png

from IPython.display import Image as IPImage, display
display(IPImage(filename=str(BUILD / 'preview.png')))

## 3. Render the corpus, then verify it

The verifier re-derives split disjointness from the written labels rather than
trusting the generator, and checks every bounding box.

In [ ]:
CORPUS = BUILD / 'corpus' / 'v1'
!python aksara_seq/scripts/02_render_corpus.py \
    --config aksara_seq/configs/corpus_v1.yaml \
    --glyph-cache {GLYPHS} --out {CORPUS}
!python aksara_seq/scripts/03_verify_corpus.py --corpus {CORPUS}

## 4. Train the matrix

Four scripts × two label spaces. Completed cells are skipped on re-run, so a
killed session resumes with the same command — which matters on Kaggle, where
the working directory does not persist between sessions unless you save it as
output.

In [ ]:
RECOG = BUILD / 'recog'
!python aksara_seq/scripts/06_run_matrix.py \
    --corpus {CORPUS} --out {RECOG} --epochs 30 --batch-size 32 --num-workers 2

In [ ]:
import csv
rows = list(csv.DictReader(open(RECOG / 'matrix.csv', encoding='utf-8')))
hdr = ['script', 'head', 'test_ser', 'test_wer', 'test_line_acc',
       'onset_error', 'vowel_error', 'ser_clean', 'ser_heavy', 'minutes']
print(' '.join(f'{h:>13s}' for h in hdr))
for r in rows:
    print(' '.join(f'{r.get(h, ""):>13s}' for h in hdr))

## 5. Optional — the detection baseline

Exports YOLO labels from the same corpus. See the caveat in the README: these
lines never overlap by construction, so detection is easier here than on real
manuscript hands.

In [ ]:
# !pip install -q ultralytics
# !python aksara_seq/scripts/05_export_yolo.py --corpus {CORPUS} --script Bali --classes syllable
# !yolo detect train data={BUILD}/yolo/Bali_syllable/data.yaml model=yolo11s.pt imgsz=960 epochs=50